In [18]:
import numpy as np
import matplotlib.pyplot as plt
import sys

In [19]:
path = "/Users/illiaderevianko/Desktop/c++/nanoparser/data/01302019.NASDAQ_ITCH50_timestamps"
ts = np.fromfile(path, dtype=np.uint64)

In [20]:
ts_sec = ts / 1e9

start_hh, start_mm = 9, 30   # set to None, None to use data min
end_hh,   end_mm   = 16, 30  # set to None, None to use data max

# compute bounds
start_sec = start_hh * 3600 + start_mm * 60 if start_hh is not None else min_sec
end_sec   = end_hh   * 3600 + end_mm   * 60 if end_hh   is not None else max_sec

# filter
mask   = (ts_sec >= start_sec) & (ts_sec <= end_sec)
ts_sec = ts_sec[mask]

min_sec = ts_sec.min()
max_sec = ts_sec.max()

min_sec = ts_sec.min()
max_sec = ts_sec.max()

bin_width_minutes = 5
bin_size = bin_width_minutes * 60
n_bins = int(np.ceil((max_sec - min_sec) / bin_size))

bin_edges = np.arange(min_sec, min_sec + (n_bins + 1) * bin_size, bin_size)

fig, ax = plt.subplots(figsize=(14, 6))
ax.hist(ts_sec, bins=bin_edges, color="#2196F3", edgecolor="white", linewidth=0.3)

# x-axis labels in hh:mm format
def sec_to_hhmm(s):
    h = int(s) // 3600
    m = (int(s) % 3600) // 60
    return f"{h:02d}:{m:02d}"

tick_positions = bin_edges[:-1] + bin_size / 2  # center of each bin
tick_labels = [sec_to_hhmm(t) for t in bin_edges[:-1]]

ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels, rotation=45, ha="right", fontsize=8)

ax.set_xlabel("Time (hh:mm)", fontsize=12)
ax.set_ylabel("Message count", fontsize=12)
ax.set_title(f"NASDAQ ITCH Message Frequency ({bin_width_minutes}-minute bins)", fontsize=14)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x):,}"))

plt.tight_layout()
plt.savefig("timestamps_histogram.png", dpi=150)
plt.show()
print(f"Total messages: {len(ts):,}")
print(f"Time range: {sec_to_hhmm(min_sec)} — {sec_to_hhmm(max_sec)}")
print(f"Bins: {n_bins} × {bin_width_minutes} minutes")

Total messages: 368,366,634
Time range: 09:30 — 16:29
Bins: 84 × 5 minutes


In [21]:
ts = np.fromfile(path, dtype=np.uint64)
ts_sec = ts / 1e9

# --- config ---
start_hh, start_mm = 9, 30   # None, None to use data min
end_hh,   end_mm   = 16, 30  # None, None to use data max
bin_width_seconds  = 1        # granularity of rate calculation

# --- filter ---
min_sec = ts_sec.min()
max_sec = ts_sec.max()
start_sec = start_hh * 3600 + start_mm * 60 if start_hh is not None else min_sec
end_sec   = end_hh   * 3600 + end_mm   * 60 if end_hh   is not None else max_sec

ts_sec = ts_sec[(ts_sec >= start_sec) & (ts_sec <= end_sec)]

# --- bin into counts per second, then compute rate ---
bin_edges = np.arange(start_sec, end_sec + bin_width_seconds, bin_width_seconds)
counts, _ = np.histogram(ts_sec, bins=bin_edges)
rate = counts / bin_width_seconds  # messages per second
bin_centers = bin_edges[:-1] + bin_width_seconds / 2

# --- format x ticks ---
def sec_to_hhmm(s):
    h = int(s) // 3600
    m = (int(s) % 3600) // 60
    return f"{h:02d}:{m:02d}"

tick_every = 10 * 60  # one tick every 10 minutes
tick_positions = np.arange(start_sec, end_sec, tick_every)
tick_labels = [sec_to_hhmm(t) for t in tick_positions]

# --- plot ---
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(bin_centers, rate, color="#2196F3", linewidth=0.6)
ax.fill_between(bin_centers, rate, alpha=0.2, color="#2196F3")

ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels, rotation=45, ha="right", fontsize=8)
ax.set_xlabel("Time (hh:mm)", fontsize=12)
ax.set_ylabel("Messages per second", fontsize=12)
ax.set_title(f"NASDAQ ITCH Message Rate ({bin_width_seconds}s bins)", fontsize=14)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x):,}"))

plt.tight_layout()
plt.savefig("message_rate.png", dpi=150)
plt.show()
print(f"Peak rate: {rate.max():,.0f} msg/s at {sec_to_hhmm(bin_centers[rate.argmax()])}")
print(f"Mean rate: {rate.mean():,.0f} msg/s")

Peak rate: 988,711 msg/s at 16:00
Mean rate: 14,166 msg/s
